# Vision

- GPT-4o with image inpus

- GPT-4o와 GPT-4 Turbo 모두 비전 기능을 갖추고 있습니다. 즉, 모델이 이미지를 촬영하고 그에 대한 질문에 답할 수 있습니다

In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

이미지는 두 가지 주요 방법, 즉 이미지에 대한 링크를 전달하거나 요청에 base64로 인코딩된 이미지를 직접 전달하는 방법으로 모델에서 사용할 수 있습니다. 

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-4o"

### 이미지의 내용 설명
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg" width=300px />

In [3]:
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

response = client.chat.completions.create(
  model=Model,
  messages=[
    {
      "role": "user",
      "content": [
        {"type": "text", "text": "이 이미지의 내용을 설명해줘. 한국어로 설명해줘."},
        {
          "type": "image_url",
          "image_url": {
            "url": url,
          },
        },
      ],
    }
  ],
  max_tokens=300,
)

response.to_dict()

{'id': 'chatcmpl-9nGxLr97JqUyJg9Yscdz2736KJumz',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '이 이미지는 자연 풍경을 담고 있습니다. 밝은 날씨에 파란 하늘과 흰 구름이 떠 있는 모습이 보이며, 중앙에는 나무로 만들어진 좁은 길이 길게 펼쳐져 있습니다. 그 양 옆으로는 푸른 풀밭이 무성하게 자라있습니다. 나무길은 멀리까지 이어져 있으며, 배경으로는 나무와 덤불이 보입니다. 이 전경은 평화롭고 아름다운 자연의 풍경을 잘 담고 있습니다.',
    'role': 'assistant'}}],
 'created': 1721530279,
 'model': 'gpt-4o-2024-05-13',
 'object': 'chat.completion',
 'system_fingerprint': 'fp_18cc0f1fa0',
 'usage': {'completion_tokens': 119,
  'prompt_tokens': 1127,
  'total_tokens': 1246}}

In [4]:
print(response.choices[0].message.content)

이 이미지는 자연 풍경을 담고 있습니다. 밝은 날씨에 파란 하늘과 흰 구름이 떠 있는 모습이 보이며, 중앙에는 나무로 만들어진 좁은 길이 길게 펼쳐져 있습니다. 그 양 옆으로는 푸른 풀밭이 무성하게 자라있습니다. 나무길은 멀리까지 이어져 있으며, 배경으로는 나무와 덤불이 보입니다. 이 전경은 평화롭고 아름다운 자연의 풍경을 잘 담고 있습니다.


### Base 64로 인코딩된 이미지 업로드
로컬에 이미지 또는 이미지 세트가 있는 경우 이를 Base 64 인코딩 형식으로 모델에 전달할 수 있습니다. 다음은 이에 대한 실제 예입니다.

In [6]:
import base64
import requests

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "data/냉장고내용물.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"
}

payload = {
  "model": Model,
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "이 이미지의 내용을 한국어로 설명해줘."
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
          }
        }
      ]
    }
  ],
  "max_tokens": 300
}

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

print(response.json())

{'id': 'chatcmpl-9nGzGmBw4hnjOtAYDmoAk7u8jXLYx', 'object': 'chat.completion', 'created': 1721530398, 'model': 'gpt-4o-2024-05-13', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '이 이미지는 냉장고 내부를 보여주고 있으며, 다양한 식품들이 잘 정돈되어 있습니다. 냉장고 안에는 다양한 색상의 야채, 과일, 음료수, 그리고 우유 등이 있습니다. 왼쪽과 오른쪽 문 선반에는 우유와 주스병, 계란, 허브 등이 놓여 있고, 중앙 선반에는 여러 종류의 신선한 채소와 과일들이 진열되어 있습니다. 하단 서랍에는 당근과 셀러리 같은 채소들이 보입니다. 전체적으로 식품들이 깔끔하게 정리되어 있는 모습입니다.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 443, 'completion_tokens': 132, 'total_tokens': 575}, 'system_fingerprint': 'fp_5e997b69d8'}


In [7]:
res = response.json()
res['choices'][0]['message']['content']

'이 이미지는 냉장고 내부를 보여주고 있으며, 다양한 식품들이 잘 정돈되어 있습니다. 냉장고 안에는 다양한 색상의 야채, 과일, 음료수, 그리고 우유 등이 있습니다. 왼쪽과 오른쪽 문 선반에는 우유와 주스병, 계란, 허브 등이 놓여 있고, 중앙 선반에는 여러 종류의 신선한 채소와 과일들이 진열되어 있습니다. 하단 서랍에는 당근과 셀러리 같은 채소들이 보입니다. 전체적으로 식품들이 깔끔하게 정리되어 있는 모습입니다.'

### 다중 이미지 입력
Chat Completions API는 base64로 인코딩된 형식이나 이미지 URL로 여러 이미지 입력을 가져와 처리할 수 있습니다. 모델은 각 이미지를 처리하고 모든 이미지의 정보를 사용하여 질문에 답합니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg" width=300px />

<img src="https://img.freepik.com/premium-photo/a-wooden-walkway-leads-to-a-green-field-with-a-mountain-in-the-background_800563-10576.jpg" width=300px />

In [8]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
  model=Model,
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "이 이미지에는 무엇이 들어있나요? 그들 사이에 어떤 차이가 있나요?",
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
          },
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://img.freepik.com/premium-photo/a-wooden-walkway-leads-to-a-green-field-with-a-mountain-in-the-background_800563-10576.jpg",
          },
        },
      ],
    }
  ],
  max_tokens=300,
)

In [9]:
response

ChatCompletion(id='chatcmpl-9nGzgB1sgO6Rhhl1mLwJP2sOesLgu', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='이 두 이미지에는 자연 풍경이 포함되어 있으며, 주된 요소는 나무로 된 길입니다. 하지만 몇 가지 차이점이 있습니다:\n\n1. **첫 번째 이미지:**\n   - 녹색 풀과 덤불이 많이 보이는 평야 지역입니다.\n   - 하늘은 맑고 적은 구름이 떠 있어 청명한 날씨를 나타냅니다.\n   - 길은 잘 정돈된 나무 판자로 되어 있습니다.\n\n2. **두 번째 이미지:**\n   - 논밭으로 보이는 녹색 필드입니다.\n   - 해가 지는 듯한 노을이 하늘에 비쳐 있으며, 분위기가 평온하고 차분해 보입니다.\n   - 다소 오래 되거나 사용된 나무 판자가 길을 이루고 있으며, 중간 중간 틈새가 보입니다.\n\n양 이미지 모두 나무로 된 길과 자연 풍경을 담고 있지만, 첫 번째 이미지는 밝고 생기 넘치는 느낌, 두 번째 이미지는 차분하고 평화로운 느낌을 줍니다.', role='assistant', function_call=None, tool_calls=None))], created=1721530424, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_5e997b69d8', usage=CompletionUsage(completion_tokens=226, prompt_tokens=1556, total_tokens=1782))

In [10]:
print(response.choices[0].message.content)

이 두 이미지에는 자연 풍경이 포함되어 있으며, 주된 요소는 나무로 된 길입니다. 하지만 몇 가지 차이점이 있습니다:

1. **첫 번째 이미지:**
   - 녹색 풀과 덤불이 많이 보이는 평야 지역입니다.
   - 하늘은 맑고 적은 구름이 떠 있어 청명한 날씨를 나타냅니다.
   - 길은 잘 정돈된 나무 판자로 되어 있습니다.

2. **두 번째 이미지:**
   - 논밭으로 보이는 녹색 필드입니다.
   - 해가 지는 듯한 노을이 하늘에 비쳐 있으며, 분위기가 평온하고 차분해 보입니다.
   - 다소 오래 되거나 사용된 나무 판자가 길을 이루고 있으며, 중간 중간 틈새가 보입니다.

양 이미지 모두 나무로 된 길과 자연 풍경을 담고 있지만, 첫 번째 이미지는 밝고 생기 넘치는 느낌, 두 번째 이미지는 차분하고 평화로운 느낌을 줍니다.
